# Logistic Regression

It's the simplest form a neural network

let's build it together with PyTorch

In [31]:
import torch
import torch.nn as nn
import numpy as np
import sklearn
from sklearn import datasets
from sklearn.preprocessing import StandardScaler  # for feature scaling
from sklearn.model_selection import train_test_split  # for train/test split

In [32]:
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
sklearn.utils.validation.check_random_state(SEED)

RandomState(MT19937) at 0x7B3D3E1D2240

In [33]:
# Prepare data
bc = datasets.load_breast_cancer()
print(bc.DESCR)

.. _breast_cancer_dataset:

Breast cancer wisconsin (diagnostic) dataset
--------------------------------------------

**Data Set Characteristics:**

:Number of Instances: 569

:Number of Attributes: 30 numeric, predictive attributes and the class

:Attribute Information:
    - radius (mean of distances from center to points on the perimeter)
    - texture (standard deviation of gray-scale values)
    - perimeter
    - area
    - smoothness (local variation in radius lengths)
    - compactness (perimeter^2 / area - 1.0)
    - concavity (severity of concave portions of the contour)
    - concave points (number of concave portions of the contour)
    - symmetry
    - fractal dimension ("coastline approximation" - 1)

    The mean, standard error, and "worst" or largest (mean of the three
    worst/largest values) of these features were computed for each image,
    resulting in 30 features.  For instance, field 0 is Mean Radius, field
    10 is Radius SE, field 20 is Worst Radius.

    - 

## Prepare data

In [34]:
X, y = bc.data, bc.target


n_samples, n_features = X.shape
print(f"number of samples: {n_samples}, number of features: {n_features}")

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1234
)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

number of samples: 569, number of features: 30


((455, 30), (114, 30), (455,), (114,))

In [35]:
# scale data
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [36]:
# convert to tensors
X_train = torch.from_numpy(X_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
y_test = torch.from_numpy(y_test.astype(np.float32))

X_train.shape, X_test.shape, y_train.shape, y_test.shape

(torch.Size([455, 30]),
 torch.Size([114, 30]),
 torch.Size([455]),
 torch.Size([114]))

In [46]:
# reshape y tensors, so they have a feature dimension

y_train = y_train.reshape(y_train.shape[0], 1)
y_test = y_test.reshape(y_test.shape[0], 1)

y_train.shape, y_test.shape

(torch.Size([455, 1]), torch.Size([114, 1]))

## Define model

In [47]:
# f = sigmoid(WX + b)
class LogisticRegression(nn.Module):

    def __init__(self, n_input_features):
        super().__init__()
        self.linear = nn.Linear(n_input_features, 1)

    def forward(self, x):
        z = self.linear(x)
        y_predicted = torch.sigmoid(z)
        return y_predicted


model = LogisticRegression(n_features)
model

LogisticRegression(
  (linear): Linear(in_features=30, out_features=1, bias=True)
)

[Visualization of a Neural Network which is Learning](https://playground.tensorflow.org/#activation=sigmoid&batchSize=10&dataset=xor&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4&seed=0.20241&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false&regularization_hide=true&discretize_hide=true&activation_hide=true&problem_hide=true&regularizationRate_hide=true&numHiddenLayers_hide=false)

## Define optimizer


In [48]:
# Loss and optimizer
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

## Define the loss_function

In [49]:
loss_function = nn.BCELoss()

# Using the GPU

In [50]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
X_train = X_train.to(device)
X_test = X_test.to(device)
y_train = y_train.to(device)
y_test = y_test.to(device)

## Training step by step

In [51]:
model.linear.weight, model.linear.weight.grad

(Parameter containing:
 tensor([[-0.1551,  0.1410,  0.0304, -0.0593,  0.1128,  0.0285,  0.1475,  0.0200,
          -0.0576,  0.0491, -0.0495,  0.0768,  0.1630,  0.1055, -0.0798,  0.1054,
           0.0327,  0.0927, -0.1113, -0.1807, -0.0705, -0.1400,  0.1498,  0.0526,
           0.0756,  0.0577, -0.0032,  0.1429, -0.1297,  0.0115]],
        device='cuda:0', requires_grad=True),
 None)

In [52]:
y_predicted = model.forward(X_train)

In [53]:
y_predicted

tensor([[0.5904],
        [0.3537],
        [0.5455],
        [0.6088],
        [0.2623],
        [0.5900],
        [0.3006],
        [0.3351],
        [0.3411],
        [0.4013],
        [0.7592],
        [0.3398],
        [0.5991],
        [0.4657],
        [0.4872],
        [0.4383],
        [0.3987],
        [0.2868],
        [0.7232],
        [0.2670],
        [0.5068],
        [0.4547],
        [0.4136],
        [0.5233],
        [0.3488],
        [0.6349],
        [0.3536],
        [0.7858],
        [0.5478],
        [0.3898],
        [0.5428],
        [0.4007],
        [0.4181],
        [0.4302],
        [0.5666],
        [0.5304],
        [0.4195],
        [0.3270],
        [0.5869],
        [0.5854],
        [0.3456],
        [0.5401],
        [0.4534],
        [0.6192],
        [0.6406],
        [0.3638],
        [0.4446],
        [0.3380],
        [0.5418],
        [0.5015],
        [0.3898],
        [0.4708],
        [0.2686],
        [0.4958],
        [0.2620],
        [0

In [54]:
loss = loss_function(y_predicted, y_train)

In [56]:
loss, loss.grad

/tmp/ipykernel_745938/2787132900.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at aten/src/ATen/core/TensorBody.h:489.)
  loss, loss.grad


(tensor(0.9781, device='cuda:0', grad_fn=<BinaryCrossEntropyBackward0>), None)

In [57]:
loss.backward()

In [59]:
model.linear.weight, model.linear.weight.grad

(Parameter containing:
 tensor([[-0.1551,  0.1410,  0.0304, -0.0593,  0.1128,  0.0285,  0.1475,  0.0200,
          -0.0576,  0.0491, -0.0495,  0.0768,  0.1630,  0.1055, -0.0798,  0.1054,
           0.0327,  0.0927, -0.1113, -0.1807, -0.0705, -0.1400,  0.1498,  0.0526,
           0.0756,  0.0577, -0.0032,  0.1429, -0.1297,  0.0115]],
        device='cuda:0', requires_grad=True),
 tensor([[ 0.4468,  0.2452,  0.4597,  0.4389,  0.2743,  0.4247,  0.4697,  0.5080,
           0.2047,  0.0509,  0.3723,  0.0032,  0.3767,  0.3751, -0.0120,  0.2305,
           0.1938,  0.2944, -0.0285,  0.0979,  0.4771,  0.2650,  0.4879,  0.4572,
           0.2972,  0.3923,  0.4330,  0.5134,  0.2283,  0.2356]],
        device='cuda:0'))

In [60]:
optimizer.step()

In [61]:
model.linear.weight, model.linear.weight.grad

(Parameter containing:
 tensor([[-0.1998,  0.1164, -0.0156, -0.1032,  0.0854, -0.0140,  0.1005, -0.0308,
          -0.0780,  0.0440, -0.0867,  0.0765,  0.1253,  0.0680, -0.0786,  0.0823,
           0.0133,  0.0633, -0.1084, -0.1905, -0.1183, -0.1665,  0.1010,  0.0069,
           0.0459,  0.0185, -0.0465,  0.0915, -0.1525, -0.0121]],
        device='cuda:0', requires_grad=True),
 tensor([[ 0.4468,  0.2452,  0.4597,  0.4389,  0.2743,  0.4247,  0.4697,  0.5080,
           0.2047,  0.0509,  0.3723,  0.0032,  0.3767,  0.3751, -0.0120,  0.2305,
           0.1938,  0.2944, -0.0285,  0.0979,  0.4771,  0.2650,  0.4879,  0.4572,
           0.2972,  0.3923,  0.4330,  0.5134,  0.2283,  0.2356]],
        device='cuda:0'))

In [62]:
optimizer.zero_grad()

In [63]:
model.linear.weight, model.linear.weight.grad

(Parameter containing:
 tensor([[-0.1998,  0.1164, -0.0156, -0.1032,  0.0854, -0.0140,  0.1005, -0.0308,
          -0.0780,  0.0440, -0.0867,  0.0765,  0.1253,  0.0680, -0.0786,  0.0823,
           0.0133,  0.0633, -0.1084, -0.1905, -0.1183, -0.1665,  0.1010,  0.0069,
           0.0459,  0.0185, -0.0465,  0.0915, -0.1525, -0.0121]],
        device='cuda:0', requires_grad=True),
 None)

# Training loop

In [42]:
num_epochs = 100

In [43]:
# training loop
for epoch in range(num_epochs):
    # forward pass and loss
    y_predicted = model.forward(X_train)
    loss = loss_function(y_predicted, y_train)

    # backward pass
    loss.backward()

    # updates
    optimizer.step()

    # zero gradients
    optimizer.zero_grad()

    if (epoch + 1) % 10 == 0:
        print(f"epoch: {epoch+1}, loss = {loss.item():.4f}")

epoch: 10, loss = 0.2440
epoch: 20, loss = 0.1749
epoch: 30, loss = 0.1454
epoch: 40, loss = 0.1282
epoch: 50, loss = 0.1167
epoch: 60, loss = 0.1083
epoch: 70, loss = 0.1018
epoch: 80, loss = 0.0966
epoch: 90, loss = 0.0923
epoch: 100, loss = 0.0887


# Evaluation

In [44]:
with torch.no_grad():
    y_train_predicted = model.forward(X_train)
    y_train_predicted_cls = y_train_predicted.round()
    acc = y_train_predicted_cls.eq(y_train).sum() / float(y_train.shape[0])  # accuracy
    print(f"accuracy train_set = {acc:.4f}")


accuracy train_set = 0.9824


In [45]:
with torch.no_grad():
    y_predicted = model.forward(X_test)
    y_predicted_cls = y_predicted.round()  # round off to nearest class
    acc = y_predicted_cls.eq(y_test).sum() / float(y_test.shape[0])  # accuracy
    print(f"accuracy test_set = {acc:.4f}")

accuracy test_set = 0.9298
